# 4.5 Tool Calling: Concepts Notebook

**Generative AI Foundations · Module 4**

| # | Concept |
|---|---|
| 1 | What tool calling is |
| 2 | Making a tool with `@tool` |
| 3 | Describing arguments (`parse_docstring=True`) |
| 4 | Giving tools to the model (`bind_tools`) |
| 5 | Reading the model's request (`tool_calls`) |
| 6 | Sending the result back (`ToolMessage`) |
| 7 | The full round trip |
| 8 | The tool loop with a step limit |
| 9 | Safety: unknown tools and tool errors |

**Model:** tool calling needs a model that supports tools. `gemma3:1b` does **not**. This notebook uses `gemma4:e2b`:

```bash
ollama pull gemma4:e2b
```

## 0. Setup

In [ ]:
%pip install -q "langchain[ollama]"

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import ToolMessage
from langchain.tools import tool

llm = init_chat_model("ollama:gemma4:e2b", temperature=0)
print("Ready")

---
## 1. What Tool Calling Is

The model can only write text. It can't check the clock or look up an order. With tools, it **writes a request**, and **your code** does the work.

```text
You:        "Convert 100 USD to PKR"
Model:      "Please run convert_currency(amount=100, from=USD, to=PKR)"     <- a request, not an answer
Your code:  runs the function -> "100 USD = 28,000 PKR"
Model:      "100 US dollars is 28,000 rupees."                              <- the answer
```

> **Mental Model: The Manager and the Assistant**
> The manager (model) can't leave the office, so it writes notes: "check order #4821". The assistant (your code) does the errand and brings back the result. The manager then writes the reply.

---
## 2. Making a Tool with `@tool`

Put `@tool` above a normal function:

| Function part | Becomes |
|---|---|
| Function name | Tool **name** |
| Docstring | Tool **description** (the model reads it to decide when to use the tool) |
| Type hints | **Argument types** |

In [ ]:
@tool
def get_order_status(order_number: str) -> str:
    """Look up the delivery status of a ShopPK order."""
    orders = {"#4821": "Delivered on 1 October.", "#5521": "Out for delivery today."}
    return orders.get(order_number, "No order found with that number.")


print("Name:       ", get_order_status.name)
print("Description:", get_order_status.description)
print("Arguments:  ", get_order_status.args)

A tool is still a normal function. You can call it yourself, which is the easiest way to test it:

In [ ]:
print(get_order_status.invoke({"order_number": "#4821"}))
print(get_order_status.invoke({"order_number": "#9999"}))

---
## 3. Describing Arguments

> **Mental Model: The Job Advert**
> The model "hires" a tool by reading its advert: the name, description, and what each argument means. A clear advert means the right tool, with the right arguments.

Compare a tool **without** and **with** `parse_docstring=True`:

In [ ]:
@tool
def convert_plain(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between currencies.

    Args:
        amount: How much money, for example 100.
        from_currency: Code to convert from, for example USD.
        to_currency: Code to convert to, for example PKR.
    """
    return ""


@tool(parse_docstring=True)
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between currencies.

    Args:
        amount: How much money, for example 100.
        from_currency: Code to convert from, for example USD.
        to_currency: Code to convert to, for example PKR.
    """
    rates = {"PKR": 1.0, "USD": 280.0, "EUR": 305.0}
    if from_currency.upper() not in rates or to_currency.upper() not in rates:
        raise ValueError(f"Unsupported currency. Use one of: {', '.join(rates)}")
    result = amount * rates[from_currency.upper()] / rates[to_currency.upper()]
    return f"{amount:,.2f} {from_currency.upper()} = {result:,.2f} {to_currency.upper()}"


print("WITHOUT:", convert_plain.args["amount"])
print("WITH:   ", convert_currency.args["amount"])

Without `parse_docstring=True`, the argument descriptions are **lost**. The model only sees the name and type.

---
## 4. Giving Tools to the Model

`bind_tools` returns a **new** model that knows about the tools. The original `llm` stays unchanged.

In [ ]:
tools = [get_order_status, convert_currency]
tools_by_name = {t.name: t for t in tools}

llm_with_tools = llm.bind_tools(tools)
print("Tools available:", list(tools_by_name))

---
## 5. Reading the Model's Request

When the model wants a tool, its reply has **empty text** and a list of **tool calls**.

In [ ]:
reply = llm_with_tools.invoke("Convert 100 USD to PKR.")

print("Text:      ", repr(reply.text))
for call in reply.tool_calls:
    print("Tool call: ", call["name"], call["args"], "id =", call["id"][:8], "...")

| Key | Meaning |
|---|---|
| `name` | Which tool to run |
| `args` | Arguments as a dictionary |
| `id` | Links the result back to this request |

A question that needs **no** tool gets a normal answer with no tool calls:

In [ ]:
reply = llm_with_tools.invoke("In one sentence, what is Python?")
print("Tool calls:", reply.tool_calls)
print("Text:      ", reply.text)

---
## 6. Sending the Result Back

Give `tool.invoke(...)` the **whole tool call**. It runs the function and returns a `ToolMessage` with the **same id**.

```text
request:  {name: convert_currency, args: {...}, id: "abc"}
result:   ToolMessage(content="100.00 USD = 28,000.00 PKR", tool_call_id="abc")
```

In [ ]:
call = {"name": "convert_currency", "args": {"amount": 100, "from_currency": "USD", "to_currency": "PKR"},
        "id": "abc123", "type": "tool_call"}

result = convert_currency.invoke(call)

print(type(result).__name__)
print("content:     ", result.content)
print("tool_call_id:", result.tool_call_id)

---
## 7. The Full Round Trip

```text
messages:  [user]                         -> model asks for a tool
           [user, ai(tool call)]          -> your code runs it
           [user, ai(tool call), tool]    -> model writes the answer
           [user, ai(tool call), tool, ai(answer)]
```

In [ ]:
messages = [{"role": "user", "content": "Convert 100 USD to PKR."}]

ai_message = llm_with_tools.invoke(messages)          # 1. model asks
messages.append(ai_message)

for call in ai_message.tool_calls:                    # 2. your code runs the tools
    messages.append(tools_by_name[call["name"]].invoke(call))

final = llm_with_tools.invoke(messages)               # 3. model answers
messages.append(final)

for m in messages:
    kind = m["role"] if isinstance(m, dict) else m.type
    text = m["content"] if isinstance(m, dict) else (m.text or f"asks for {[c['name'] for c in m.tool_calls]}")
    print(f"[{kind:<4}] {text}")

---
## 8. The Tool Loop with a Step Limit

Sometimes the model needs **several** rounds of tools. So we loop until it answers without tool calls, but never more than `MAX_STEPS` times.

> **Mental Model: The Errand Runner with a Budget**
> "Do as many errands as you need, but at most five trips."

```text
repeat up to MAX_STEPS:
    reply = model(messages)
    if no tool calls -> done
    run each tool, add results
```

In [ ]:
MAX_STEPS = 5

def ask(question):
    messages = [{"role": "user", "content": question}]
    for step in range(1, MAX_STEPS + 1):
        reply = llm_with_tools.invoke(messages)
        messages.append(reply)
        if not reply.tool_calls:
            return reply.text
        for call in reply.tool_calls:
            result = tools_by_name[call["name"]].invoke(call)
            messages.append(result)
            print(f"  step {step}: {call['name']} -> {result.content}")
    return "Stopped: too many steps."


for question in ["What is the status of order #5521?", "Convert 50 USD to PKR.", "What is an API? One sentence."]:
    print("You:", question)
    print("Cue:", ask(question), "\n")

---
## 9. Safety: Unknown Tools and Tool Errors

Two things can go wrong **in your code**:

| Problem | Example | Safe handling |
|---|---|---|
| The model invents a tool | `launch_rocket` | Reply with an error `ToolMessage` |
| The tool fails | Unknown currency `JPY` | Catch it; reply with an error `ToolMessage` |

The model can read an error message and explain it to the user. A crash helps nobody.

In [ ]:
def run_tool(call):
    tool = tools_by_name.get(call["name"])
    if tool is None:
        return ToolMessage(f"Error: unknown tool '{call['name']}'.", tool_call_id=call["id"], status="error")
    try:
        return tool.invoke(call)
    except Exception as error:
        return ToolMessage(f"Error: {error}", tool_call_id=call["id"], status="error")


bad_calls = [
    {"name": "launch_rocket", "args": {}, "id": "t1", "type": "tool_call"},
    {"name": "convert_currency", "args": {"amount": 5, "from_currency": "JPY", "to_currency": "PKR"}, "id": "t2", "type": "tool_call"},
]

for call in bad_calls:
    result = run_tool(call)
    print(f"{call['name']:<17} -> status={result.status}: {result.content}")

---
## Summary

| Concept | Remember |
|---|---|
| Tool calling | The model **requests**, your code **runs** |
| `@tool` | Name, docstring, and type hints become the tool's "advert" |
| `parse_docstring=True` | Sends argument descriptions too |
| `bind_tools` | A new model that knows the tools |
| `reply.tool_calls` | Requests with `name`, `args`, `id` |
| `tool.invoke(call)` | Runs the tool, returns a `ToolMessage` with the matching id |
| Tool loop | Repeat until no tool calls; cap with `MAX_STEPS` |
| Safety | Unknown tools and errors become error messages, not crashes |

## Practice

1. Write a `get_current_time` tool, add it to `tools`, and ask: "What time is it, and what's the status of order #4821?"
2. In Section 8, set `MAX_STEPS = 1` and ask the same questions. What changes?
3. In Section 9, add a check that refuses `amount` values below 0 inside `convert_currency`, and test it with `run_tool`.